# Create pollutant and overall air-quality labels from cleaned readings

This notebook applies the thresholds below to `cleaned_dataset_validated.csv`, produced by `clean_readings.ipynb`. It assigns healthy, moderate, and unhealthy labels to each pollutant and an overall label based on the worst available pollutant. MySQL is not needed unless you explicitly enable saving the train/test splits.


## WHO basis and the three-class policy

This notebook uses the supplied WHO guideline values as the healthy-to-moderate boundaries. Official sources: [WHO 2021 guidelines](https://www.who.int/publications/i/item/9789240034228) and [WHO air-quality-guideline overview](https://www.who.int/news-room/feature-stories/detail/what-are-the-who-air-quality-guidelines).

WHO publishes guideline values and interim targets, but it does not define one universal set of labels named healthy, moderate, and unhealthy. For this project, use these operational rules:

- **Healthy (0):** value is at or below the supplied guideline.
- **Moderate (1):** value is above the guideline but at or below the configured upper cutoff.
- **Unhealthy (2):** value is above the configured upper cutoff.

| Pollutant | Averaging period | Healthy through | Moderate through (upper cutoff) | Threshold unit |
| --- | --- | ---: | ---: | --- |
| PM2.5 | 24 hours | 15 | 75 | µg/m³ |
| PM10 | 24 hours | 45 | 150 | µg/m³ |
| NO₂ | 24 hours | 25 | 120 | µg/m³ |
| SO₂ | 24 hours | 40 | 125 | µg/m³ |
| CO | 24 hours | 4 | 7 | mg/m³ |
| O₃ | trailing 8 hours | 100 | 160 | µg/m³ |

Calculate trailing rolling means separately for each station before classification: 24 hours for PM2.5, PM10, NO2, SO2, and CO; 8 hours for O3. Require a complete window of valid hourly readings (24 of 24 or 8 of 8); otherwise the rolling average and its class are missing. The supplied values define the guideline boundary; the configured upper cutoffs are project choices. The previous CO upper cutoff of 35 mg/m³ was for an 8-hour average, so use the WHO 2005 24-hour value of 7 mg/m³ instead. The Beijing dataset stores CO in µg/m³; the code converts rolling averages to mg/m³ for comparison while preserving the original column. These are screening labels, not formal WHO compliance determinations.

The overall verdict is the **maximum severity among all valid pollutant labels**: any unhealthy pollutant makes the overall result unhealthy; otherwise, any moderate pollutant makes it moderate; only when every available pollutant is healthy is the overall result healthy. Missing pollutant values are ignored, and a row with no valid pollutant values receives no overall label.


## Create reproducible stratified training and test sets

An 80/20 split is made separately within each overall label category so both sets retain approximately the same class proportions. Rows with no valid pollutant measurement are excluded because they cannot receive an overall verdict. The sets are created from the cleaned readings and in-memory labels; no classifier or model is created.


In [30]:
from pathlib import Path
import os
import pandas as pd

WRITE_SPLITS_TO_DB = False
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "Notebooks",
    Path.cwd() / "Data_Science" / "Notebooks",
]
DATA_DIR = next(
    (directory for directory in candidate_dirs
     if (directory / "cleaned_dataset_validated.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find cleaned_dataset_validated.csv in the current directory "
        "or its expected Notebooks subdirectories. Run clean_readings.ipynb first."
    )

SOURCE_FILE = DATA_DIR / "cleaned_dataset_validated.csv"
source_df = pd.read_csv(SOURCE_FILE)
print(f"Loaded {len(source_df):,} cleaned rows from {SOURCE_FILE}")
display(source_df.head())


Loaded 418,753 cleaned rows from c:\Users\PLP\Downloads\Data_Science\Data_Science\Notebooks\cleaned_dataset_validated.csv


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,interpolated_columns
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,NaN
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,NaN
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,NaN
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,NaN
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,NaN


## Calculate rolling averages and label the pollutants

Each pollutant is classified from its station-specific trailing rolling average: 24 hours for PM2.5, PM10, NO2, SO2, and CO; 8 hours for O3. Every hour in the window must have a valid reading. Each pollutant receives a readable label and numeric code, and `overall_health_label` is the worst available pollutant code. CO averages are converted from µg/m³ to mg/m³ for comparison.


In [31]:
WHO_THRESHOLDS = {
    "PM2.5": {"guideline": 15.0, "upper_cutoff": 75.0, "unit": "ug/m3", "period": "24-hour"},
    "PM10": {"guideline": 45.0, "upper_cutoff": 150.0, "unit": "ug/m3", "period": "24-hour"},
    "NO2": {"guideline": 25.0, "upper_cutoff": 120.0, "unit": "ug/m3", "period": "24-hour"},
    "SO2": {"guideline": 40.0, "upper_cutoff": 125.0, "unit": "ug/m3", "period": "24-hour"},
    "CO": {"guideline": 4.0, "upper_cutoff": 7.0, "unit": "mg/m3", "period": "24-hour", "source_to_threshold_factor": 0.001},
    "O3": {"guideline": 100.0, "upper_cutoff": 160.0, "unit": "ug/m3", "period": "8-hour"},
}
LABEL_CODES = {"healthy": 0, "moderate": 1, "unhealthy": 2}

required_columns = [*WHO_THRESHOLDS, "station", "year", "month", "day", "hour"]
missing_columns = [column for column in required_columns if column not in source_df.columns]
if missing_columns:
    raise KeyError(f"Required pollutant columns are missing from {SOURCE_FILE.name!r}: {missing_columns}")

labeled_df = source_df.copy()
time_parts = labeled_df[["year", "month", "day", "hour"]].apply(pd.to_numeric, errors="coerce")
timestamps = pd.to_datetime(
    {"year": time_parts["year"], "month": time_parts["month"], "day": time_parts["day"], "hour": time_parts["hour"]},
    errors="coerce",
)
station_keys = labeled_df["station"].astype("string").str.strip().str.casefold()
valid_time = timestamps.notna() & station_keys.notna()
time_data = labeled_df.loc[valid_time, ["station"]].copy()
time_data["_timestamp"] = timestamps.loc[valid_time]
time_data["_station_key"] = station_keys.loc[valid_time]
rolling_hours = {pollutant: 24 for pollutant in WHO_THRESHOLDS}
rolling_hours["O3"] = 8
for pollutant, window_hours in rolling_hours.items():
    readings = pd.to_numeric(labeled_df[pollutant], errors="coerce")
    readings = readings.mask(readings.lt(0))
    rolling_average = pd.Series(float("nan"), index=labeled_df.index, dtype="float64")
    min_readings = window_hours
    for _, station_rows in time_data.groupby("_station_key", sort=False):
        ordered = station_rows.sort_values("_timestamp")
        time_indexed_readings = pd.Series(
            readings.loc[ordered.index].to_numpy(),
            index=pd.DatetimeIndex(ordered["_timestamp"]),
        )
        rolling_mean = time_indexed_readings.rolling(
            f"{window_hours}h", closed="right", min_periods=min_readings
        ).mean()
        rolling_average.loc[ordered.index] = rolling_mean.to_numpy()
    labeled_df[f"{pollutant}_{window_hours}h_avg"] = rolling_average

for pollutant, thresholds in WHO_THRESHOLDS.items():
    labeled_df[pollutant] = pd.to_numeric(labeled_df[pollutant], errors="coerce")
    value = labeled_df[f"{pollutant}_{rolling_hours[pollutant]}h_avg"]
    value = value * thresholds.get("source_to_threshold_factor", 1.0)
    label_column = f"{pollutant}_health_label"
    code_column = f"{pollutant}_health_code"
    labeled_df[label_column] = "unhealthy"
    labeled_df.loc[value.le(thresholds["upper_cutoff"]), label_column] = "moderate"
    labeled_df.loc[value.le(thresholds["guideline"]), label_column] = "healthy"
    labeled_df.loc[value.isna(), label_column] = pd.NA
    labeled_df[code_column] = labeled_df[label_column].map(LABEL_CODES).astype("Int64")

code_columns = [f"{pollutant}_health_code" for pollutant in WHO_THRESHOLDS]
labeled_df["overall_health_code"] = labeled_df[code_columns].max(axis=1, skipna=True).astype("Int64")
reverse_codes = {code: label for label, code in LABEL_CODES.items()}
labeled_df["overall_health_label"] = labeled_df["overall_health_code"].map(reverse_codes)
print("Pollutant label counts:")
for pollutant in WHO_THRESHOLDS:
    labels = labeled_df[f"{pollutant}_health_label"]
    counts = labels.value_counts().reindex(list(LABEL_CODES), fill_value=0)
    print(f"{pollutant}:\n{counts.to_string()}")
print("\nOverall label counts:")
overall_counts = labeled_df["overall_health_label"].value_counts().reindex(list(LABEL_CODES), fill_value=0)
display(overall_counts.rename("rows"))


Pollutant label counts:
PM2.5:
PM2.5_health_label
healthy       43054
moderate     202217
unhealthy    167592
PM10:
PM10_health_label
healthy       90063
moderate     238963
unhealthy     85893
NO2:
NO2_health_label
healthy       69487
moderate     330546
unhealthy      9620
SO2:
SO2_health_label
healthy      372626
moderate      38675
unhealthy       743
CO:
CO_health_label
healthy      391249
moderate       8472
unhealthy       972
O3:
O3_health_label
healthy      345378
moderate      45605
unhealthy     22022

Overall label counts:


overall_health_label
healthy       26008
moderate     208605
unhealthy    182922
Name: rows, dtype: int64

## Create reproducible stratified training and test sets

An 80/20 split is made separately within each overall label category so both sets retain approximately the same class proportions. Rows with no valid pollutant measurement are excluded because they cannot receive an overall verdict. The sets are created from the cleaned readings and in-memory labels; no classifier or model is created.


In [34]:
TEST_FRACTION = 0.20
RANDOM_STATE = 42
model_ready_df = labeled_df.dropna(subset=["overall_health_code"]).copy()
train_parts = []
test_parts = []
for _, group in model_ready_df.groupby("overall_health_label", sort=False):
    test_count = max(1, round(len(group) * TEST_FRACTION))
    test_part = group.sample(n=test_count, random_state=RANDOM_STATE)
    train_parts.append(group.drop(index=test_part.index))
    test_parts.append(test_part)

train_set = pd.concat(train_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_set = pd.concat(test_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Training set: {len(train_set):,} rows")
print(f"Test set: {len(test_set):,} rows")
display(train_set["overall_health_label"].value_counts(normalize=True).sort_index().rename("training_proportion"))
display(test_set["overall_health_label"].value_counts(normalize=True).sort_index().rename("test_proportion"))


Training set: 334,028 rows
Test set: 83,507 rows


overall_health_label
healthy      0.062288
moderate     0.499611
unhealthy    0.438101
Name: training_proportion, dtype: float64

overall_health_label
healthy      0.062294
moderate     0.499611
unhealthy    0.438095
Name: test_proportion, dtype: float64

## Optionally save the split tables in MySQL

The default is `False` to avoid changing the database unexpectedly. Set `WRITE_SPLITS_TO_DB = True` after reviewing the generated counts. The tables contain the original pollutant values, every pollutant-specific label, and the overall label. Use versioned table names instead if multiple experiments must be retained.


In [33]:
if WRITE_SPLITS_TO_DB:
    DB_URL = os.environ.get("AIR_QUALITY_DB_URL")
    if not DB_URL:
        raise RuntimeError("Set AIR_QUALITY_DB_URL before enabling database writes.")
    from sqlalchemy import create_engine
    engine = create_engine(DB_URL)
    train_set.to_sql("training_set", engine, if_exists="replace", index=False)
    test_set.to_sql("test_set", engine, if_exists="replace", index=False)
    print("Saved training_set and test_set tables to air_quality")
else:
    print("Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.")


Database writes disabled. Set WRITE_SPLITS_TO_DB = True to save the split tables.
